In [ ]:
# Install & Import
!pip install shap lime scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries


In [ ]:
# Load & Preprocess MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1) / 255.0
X_test  = X_test.reshape(-1, 28, 28, 1) / 255.0
print(f"Train: {X_train.shape}, Test: {X_test.shape}, Classes: {len(np.unique(y_train))}")

# Show sample digits
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(f"Digit: {y_train[i]}"); ax.axis('off')
plt.show()


In [ ]:
# Build & Train CNN
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(X_train, y_train, epochs=2, validation_split=0.2, verbose=1)
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc:.4f}")


In [ ]:
# SHAP Explainability
explainer = shap.DeepExplainer(model, X_train[:100])
shap_values = explainer.shap_values(X_test[:1])
print("SHAP Explanation Generated")
shap.image_plot(shap_values, X_test[:1])


In [ ]:
# LIME Explainability
explainer_lime = lime_image.LimeImageExplainer()
img = np.repeat(X_test[0], 3, axis=2)  # grayscale → RGB

def predict_fn(images):
    gray = np.array(images)[:,:,:,0].reshape(-1,28,28,1)
    return model.predict(gray, verbose=0)

explanation = explainer_lime.explain_instance(
    img.astype('double'), predict_fn, top_labels=1, hide_color=0, num_samples=1000
)
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False
)

plt.figure(figsize=(5,5))
plt.imshow(mark_boundaries(temp, mask))
plt.title("LIME Explanation"); plt.axis('off'); plt.show()
print("LIME Explanation Generated")


In [ ]:
# GAN: Generate Handwritten Digits
(X_gan, _), _ = mnist.load_data()
X_gan = (X_gan.reshape(-1, 28, 28, 1).astype("float32") - 127.5) / 127.5

# Generator & Discriminator
generator = Sequential([
    layers.Dense(128, activation='relu', input_shape=(100,)),
    layers.Dense(28*28, activation='tanh'),
    layers.Reshape((28, 28, 1))
])
discriminator = Sequential([
    layers.Flatten(input_shape=(28,28,1)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
discriminator.compile(optimizer='adam', loss='binary_crossentropy')

discriminator.trainable = False
gan = Sequential([generator, discriminator])
gan.compile(optimizer='adam', loss='binary_crossentropy')

# Train GAN
batch_size = 64
for epoch in range(1000):
    idx  = tf.random.uniform([batch_size], 0, X_gan.shape[0], dtype=tf.int32)
    real = tf.gather(X_gan, idx)
    fake = generator.predict(tf.random.normal([batch_size, 100]), verbose=0)
    discriminator.train_on_batch(real, tf.ones((batch_size, 1)))
    discriminator.train_on_batch(fake, tf.zeros((batch_size, 1)))
    gan.train_on_batch(tf.random.normal([batch_size, 100]), tf.ones((batch_size, 1)))

# Show generated digit
img_gen = generator.predict(tf.random.normal([1, 100]), verbose=0)[0,:,:,0]
plt.imshow(img_gen, cmap='gray'); plt.axis('off')
plt.title("GAN Generated Digit"); plt.show()
